# Name That LLM – Abgabe: TF-IDF + Stilometrie + Logistische Regression

CV (GroupKFold nach `prompt_id`): **Balanced Accuracy ≈ 0.352**. Das Notebook trainiert auf `train.csv` und sagt `test.csv` vorher. Interaktiv enthält `test.csv` nur eine Beispielzeile; beim Einreichen läuft das Notebook erneut auf dem versteckten Test (~4000 Texte). Ausgabe: `submission.parquet` im Arbeitsverzeichnis. Internet wird nicht benötigt.

In [1]:
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Kaggle-Pfad laut "Submission File"-Seite; Fallback: Suche unter /kaggle/input, lokal ../data
KAGGLE_DIR = Path("/kaggle/input/competitions/name-that-llm")
if KAGGLE_DIR.exists():
    DATA_DIR = KAGGLE_DIR
else:
    candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    DATA_DIR = Path(candidates[0]).parent if candidates else Path(os.environ.get("NTL_DATA_DIR", "../data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)
print(sorted(p.name for p in DATA_DIR.iterdir()))

DATA_DIR: ../data | OUT_DIR: .
['.gitkeep', 'sample_submission.parquet', 'test.csv', 'train.csv']


## Features (aus `src/features.py`)

In [2]:
import re

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import StandardScaler

# Zeichen, deren Häufigkeit (pro 100 Wörter) als Feature dient
CHAR_COUNTS = {
    "em_dash": "—",
    "en_dash": "–",
    "hyphen": "-",
    "curly_apos": "’",
    "curly_dquote": "“",
    "straight_apos": "'",
    "straight_dquote": '"',
    "colon": ":",
    "semicolon": ";",
    "comma": ",",
    "paren": "(",
    "question": "?",
    "exclam": "!",
    "ellipsis": "…",
}

_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
_WORD = re.compile(r"\b\w+\b")


def stylometric_features(texts: pd.Series) -> pd.DataFrame:
    """Längen-, Satz-, Wort- und Interpunktionsstatistiken je Text."""
    rows = []
    for text in texts:
        words = _WORD.findall(text)
        n_words = max(len(words), 1)
        sents = [s for s in _SENT_SPLIT.split(text.strip()) if s]
        sent_lens = [len(_WORD.findall(s)) for s in sents] or [0]
        row = {
            "n_chars": len(text),
            "n_words": len(words),
            "n_sents": len(sents),
            "n_paras": text.count("\n\n") + 1,
            "words_per_sent": np.mean(sent_lens),
            "sent_len_std": np.std(sent_lens),
            "avg_word_len": np.mean([len(w) for w in words]) if words else 0,
            "type_token_ratio": len({w.lower() for w in words}) / n_words,
            "upper_ratio": sum(c.isupper() for c in text) / max(len(text), 1),
            "non_ascii_ratio": sum(ord(c) > 127 for c in text) / max(len(text), 1),
            "digit_ratio": sum(c.isdigit() for c in text) / max(len(text), 1),
            "ends_with_period": float(text.rstrip().endswith(".")),
        }
        for name, ch in CHAR_COUNTS.items():
            row[f"per100w_{name}"] = 100 * text.count(ch) / n_words
        rows.append(row)
    return pd.DataFrame(rows, index=texts.index)


class StyloTransformer(BaseEstimator, TransformerMixin):
    """sklearn-Wrapper um stylometric_features (für Pipelines)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return stylometric_features(pd.Series(X)).to_numpy(dtype=float)


def word_tfidf(**kw) -> TfidfVectorizer:
    params = dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=False)
    return TfidfVectorizer(**{**params, **kw})


def char_tfidf(**kw) -> TfidfVectorizer:
    params = dict(analyzer="char", ngram_range=(1, 5), min_df=3, sublinear_tf=True, lowercase=False)
    return TfidfVectorizer(**{**params, **kw})


def text_features(use_stylo: bool = True):
    """FeatureUnion aus Wort-TF-IDF, Char-TF-IDF und (optional) skalierten Stilo-Features."""
    parts = [word_tfidf(), char_tfidf()]
    if use_stylo:
        parts.append(make_pipeline(StyloTransformer(), StandardScaler()))
    return make_union(*parts)

## Training & Vorhersage

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print("train", train.shape, "| test", test.shape)

model = make_pipeline(text_features(use_stylo=True), LogisticRegression(C=10, max_iter=5000))
model.fit(train["text"], train["model"])
pred = model.predict(test["text"])

train (2699, 4) | test (3, 2)


## Submission
`submission.parquet` mit den Spalten `id`, `model`. Die Prüfungen unten entsprechen den Ablehnungsgründen von Kaggle: fehlende Spalten, Nullwerte, doppelte IDs, unbekannte Labels.

In [4]:
sub = pd.DataFrame({"id": test["id"], "model": pred})

# Prüfungen = Ablehnungsgründe von Kaggle
assert list(sub.columns) == ["id", "model"]
assert sub.notna().all().all(), "Nullwerte in der Submission"
assert sub["id"].is_unique, "doppelte IDs"
assert len(sub) == len(test) and set(sub["id"]) == set(test["id"]), "nicht jede Test-ID genau einmal"
assert train["model"].nunique() == 18 and set(sub["model"]) <= set(train["model"]), "unbekanntes Label"

sub.to_parquet(OUT_DIR / "submission.parquet", index=False)
print(sub["model"].value_counts().head(10))
sub.head()

model
gpt-5.6-luna              1
gpt-5.6-sol               1
gemini-3.1-pro-preview    1
Name: count, dtype: int64


,id,model
0,00353fb1c541,gpt-5.6-luna
1,005f5a9a6139,gpt-5.6-sol
2,0062a53d5487,gemini-3.1-pro-preview
